In [ ]:
# Colab'da Runtime > Change runtime type > T4 GPU secin.
import os
import torch

assert torch.cuda.is_available(), "Colab'da GPU runtime secin."
os.environ["HF_HOME"] = "/content/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/content/huggingface/transformers"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = "/content/huggingface/sentence_transformers"
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4
HF_TOKEN bulunamadi; model yukleme hucrelerinde notebook_login() ile giris yapilacak.


In [ ]:
from google.colab import drive
import os
import subprocess
import sys


drive.mount("/content/drive")

# Ilk kullanimda kendi GitHub repository adresinizi yazin.
REPO_URL = "https://github.com/aykutpt/hukuk_rag.git"
REPO_DIR = "/content/hukuk_rag_src"
if "KULLANICI" in REPO_URL:
    raise ValueError("REPO_URL satirina kendi GitHub adresinizi yazin.")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")],
    check=True,
)
sys.path.insert(0, REPO_DIR)
from dotenv import load_dotenv
from rag import HukukRAG

PROJECT_DIR = "/content/drive/MyDrive/hukuk_rag"
PDF_DIR = os.path.join(PROJECT_DIR, "pdf")
VECTOR_DB_DIR = os.path.join(PROJECT_DIR, "vector_db")
ENV_PATH = os.path.join(PROJECT_DIR, ".env")
CACHE_DIR = "/content/huggingface"
load_dotenv(ENV_PATH, override=True)
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
assert HF_TOKEN, f"Drive'da .env icinde HF_TOKEN yok: {ENV_PATH}"
assert os.path.isdir(PDF_DIR), f"PDF klasoru bulunamadi: {PDF_DIR}"
print("Guncel rag.py GitHub'dan alindi ve Drive hazir.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bulunan PDF sayisi: 11


In [ ]:
rag = HukukRAG(
    pdf_dir=PDF_DIR,
    vector_db_dir=VECTOR_DB_DIR,
    cache_dir=CACHE_DIR,
)

index_file = os.path.join(VECTOR_DB_DIR, "hukuk.index")
if os.path.isfile(index_file):
    rag.load_index()
    print(f"Mevcut indeks yuklendi: {rag.index.ntotal} parca")
else:
    count = rag.build_index()
    print(f"Yeni indeks olusturuldu: {count} parca")

rag.load_llama(HF_TOKEN)
print("Llama modeli hazir.")

In [ ]:
question = "Sözleşmeden doğan borcun ifa edilmemesinin sonuçları nelerdir?"
answer, sources = rag.ask(question, top_k=5)
print(answer)
print("\nKaynaklar:")
for source in sources:
    print(f"- {source['source']} / sayfa {source['page']} / skor {source['score']:.3f}")